# 0.0. Setup

In [1]:
import os
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(PROJECT_DIR)

DATA_DIR = 'data'
DATA_PUBLIC_DIR = 'data/public'
DATA_PRIVATE_DIR = 'data/private'
DATA_PROCESSED_DIR = 'data/processed'
OUTPUT_DIR = 'output'

## 0.1. Module

In [2]:
from modules import subsidy_tuition_processor as stp

# 1.0. Load data

In [3]:
# Initialize processor
processor_stp = stp.SubsidyTuitionProcessor(verbose=False)

# Process both datasets
esc_data, shsvp_data = processor_stp.process()

# Get ESC summary
esc_summary = processor_stp.get_esc_summary()

/workspace/project_paaral/modules/subsidy_tuition_processor.py:526: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  'amounts_by_grade': self.processed_esc_data.groupby('grade_level')['amount'].sum().to_dict(),
/workspace/project_paaral/modules/subsidy_tuition_processor.py:527: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  'amounts_by_fee_type': self.processed_esc_data.groupby('fee_type')['amount'].sum().to_dict()


## 1.1. ESC

In [4]:
display(esc_data.head())

,school_id,grade_level,fee_type,amount,ESC School ID,School Name,DepEd School Id.1,Region,Province,City/Municipality,Division
0,400001,G7,Tuition,13353.05,100008,Saint Andrew Academy,400001,Region I,Ilocos Norte,Bacarra,Ilocos Norte
1,400001,G8,Tuition,13352.34,100008,Saint Andrew Academy,400001,Region I,Ilocos Norte,Bacarra,Ilocos Norte
2,400001,G9,Tuition,13352.16,100008,Saint Andrew Academy,400001,Region I,Ilocos Norte,Bacarra,Ilocos Norte
3,400001,G10,Tuition,13252.51,100008,Saint Andrew Academy,400001,Region I,Ilocos Norte,Bacarra,Ilocos Norte
4,400001,G7,Other,3817.85,100008,Saint Andrew Academy,400001,Region I,Ilocos Norte,Bacarra,Ilocos Norte


In [17]:
# Process a little bit to remove school info
pvt_esc = esc_data.pivot_table(
    values='amount',
    index=['school_id'],
    columns='grade_level',
    aggfunc='sum',
    observed=False
)

pvt_esc.columns = ['total_fees_grade_' + re.findall(r'G(\d)', grade)[0] for grade in pvt_esc.columns]
pvt_esc['average_jhs_total_fees'] = np.round(pvt_esc.mean(axis=1), 2)

pvt_esc.reset_index(inplace=True)
pvt_esc['school_id'] = pvt_esc['school_id'].astype(str)
# pvt_esc['ESC School ID'] = pvt_esc['ESC School ID'].astype(str)
# pvt_esc.rename(
#     columns={'ESC School ID':'esc_school_id'},
#     inplace=True
# )

In [18]:
print(pvt_esc.shape)
display(pvt_esc.head())

(3621, 6)


,school_id,total_fees_grade_7,total_fees_grade_8,total_fees_grade_9,total_fees_grade_1,average_jhs_total_fees
0,400001,19390.90,19571.19,19357.93,19263.84,19395.96
1,400002,10717.24,10971.34,11086.84,11162.08,10984.38
2,400003,14023.66,14354.86,14809.29,15213.94,14600.44
3,400004,11524.86,11804.22,11971.84,12195.33,11874.06
4,400006,14363.89,13940.82,12594.67,12594.46,13373.46


In [9]:
pvt_esc.dtypes

school_id                  object
esc_school_id              object
total_fees_grade_7        float64
total_fees_grade_8        float64
total_fees_grade_9        float64
total_fees_grade_1        float64
average_jhs_total_fees    float64
dtype: object

## 1.2. SHSVP

In [39]:
display(shsvp_data.head(2))

,School ID,School Name,Region,Province,City/Municipality,Division,Type,Grade Level,Active,Track,Strand,Tuition,Other,Miscellaneous,Total Fees,school_id
0,324731,Balabagan Trade School,BARMM,Lanao del Sur,Balabagan,Lanao del Sur II,SUC/LUC,Grade 11,NaN,Academic Track,General Academic Strand,8000.0,0.0,1700.0,9700.0,324731
1,324731,Balabagan Trade School,BARMM,Lanao del Sur,Balabagan,Lanao del Sur II,SUC/LUC,Grade 11,NaN,Technical-Vocational-Livelihood Track,Electrical Installation and Maintenance (NC II),8000.0,0.0,1700.0,9700.0,324731


In [43]:
relevant_cols = ['school_id','Total Fees', 'Grade Level']

pvt_shsvp = shsvp_data[relevant_cols].pivot_table(
    values='Total Fees',
    index='school_id',
    columns='Grade Level',
    observed=False,
    aggfunc='sum'
)

pvt_shsvp.columns = ['total_fees_grade_' + str(re.findall(r'Grade (\d.)', grade)[0]) for grade in pvt_shsvp.columns]
pvt_shsvp['average_shs_total_fees'] = pvt_shsvp.mean(axis=1)

pvt_shsvp.reset_index(inplace=True)
pvt_shsvp['school_id'] = pvt_shsvp['school_id'].astype(str)

In [44]:
print(pvt_shsvp.shape)
display(pvt_shsvp.head())

(4527, 4)


,school_id,total_fees_grade_11,total_fees_grade_12,average_shs_total_fees
0,324731,19400.0,20400.0,19900.0
1,400001,20945.0,20945.0,20945.0
2,400002,70632.0,70632.0,70632.0
3,400003,379080.0,379080.0,379080.0
4,400004,37500.0,37500.0,37500.0


In [41]:
pvt_shsvp.dtypes

school_id                  object
total_fees_grade_11       float64
total_fees_grade_12       float64
average_shs_total_fees    float64
dtype: object

## 1.3. Consolidate

In [45]:
rel_esc_cols = ['school_id','average_jhs_total_fees']
rel_shs_cols = ['school_id','average_shs_total_fees']

df_mrg = pvt_esc[rel_esc_cols].merge(
    pvt_shsvp[rel_shs_cols],
    left_on='school_id',
    right_on='school_id',
    how='outer'
)
print(df_mrg.shape)

(5188, 3)


In [46]:
display(df_mrg)

,school_id,average_jhs_total_fees,average_shs_total_fees
0,324731,NaN,19900.0
1,400001,19395.96,20945.0
2,400002,10984.38,70632.0
3,400003,14600.44,379080.0
4,400004,11874.06,37500.0
...,...,...,...
5183,600342,NaN,15000.0
5184,600343,NaN,28350.0
5185,600348,NaN,107500.0
5186,600372,NaN,44250.0


In [55]:
esc_ids = pvt_esc['school_id'].unique().tolist()
shsvp_ids = pvt_shsvp['school_id'].unique().tolist()

mask_esc = df_mrg['school_id'].isin(esc_ids)
df_mrg.loc[mask_esc, 'is_esc_delivering'] = True
df_mrg['is_esc_delivering'] = df_mrg['is_esc_delivering'].fillna(False)

mask_shs = df_mrg['school_id'].isin(shsvp_ids)
df_mrg.loc[mask_shs, 'is_shsvp_delivering'] = True
df_mrg['is_shsvp_delivering'] = df_mrg['is_shsvp_delivering'].fillna(False)

In [56]:
display(df_mrg)

,school_id,average_jhs_total_fees,average_shs_total_fees,is_esc_delivering,is_shsvp_delivering
0,324731,NaN,19900.0,False,True
1,400001,19395.96,20945.0,True,True
2,400002,10984.38,70632.0,True,True
3,400003,14600.44,379080.0,True,True
4,400004,11874.06,37500.0,True,True
...,...,...,...,...,...
5183,600342,NaN,15000.0,False,True
5184,600343,NaN,28350.0,False,True
5185,600348,NaN,107500.0,False,True
5186,600372,NaN,44250.0,False,True


In [57]:
df_mrg['is_esc_delivering'].value_counts()

is_esc_delivering
True     3621
False    1567
Name: count, dtype: int64

In [58]:
df_mrg['is_shsvp_delivering'].value_counts()

is_shsvp_delivering
True     4527
False     661
Name: count, dtype: int64

### Save

In [59]:
fname = 'processed_gastpe_data.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_mrg.to_parquet(outpath)